# Button pressing and person reaching

Loads the runs collected into `outputs/button_pressing/` and `outputs/person_avoiding/`
by `collect_run.py`, and extracts the steady-state window of each one automatically.

Not yet run against real data -- the loading and plotting code below is smoke-tested
against synthetic fixtures only. Check the sign of `force` before trusting the force axes:
`arduino_force.py` publishes `-force_shear`.


In [ ]:
import sys, os, importlib.util
sys.path.insert(0, os.path.abspath('..'))
sys.path.insert(0, os.path.abspath('../Kinematics'))
from plot_config import COLORS, FIG_W_SINGLE, FIG_W_DOUBLE, set_font_size
from FK_Multi import FK_Multi
GRID = False
FONT_SIZE = 18
set_font_size(FONT_SIZE)

import numpy as np
import matplotlib.pyplot as plt

G_TO_N = 9.80665e-3   # load cell reports grams
S_BACKBONE = np.linspace(0.0, 1.0, 25)


In [ ]:
def load_cfg(name):
    """Import a config variant by file name without copying it into place."""
    spec = importlib.util.spec_from_file_location('cfg_' + name, f'GUI_MultiVMC_Config_{name}.py')
    mod  = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod


def load_run(folder, name):
    """Load one run and mark its steady-state window.

    The hold phase is where the tip goal sits at its farthest point AND the
    Gaussian strength is at its maximum -- that excludes approach and retract
    without needing any hardcoded timing.
    """
    d = np.load(os.path.join('outputs', folder, name + '.npz'), allow_pickle=True)
    tip_goal = d['goal'][:, 0, :]
    k_gauss  = d['k_params'][:, 1]

    press_target = tip_goal[np.argmax(np.linalg.norm(tip_goal - tip_goal[0], axis=1))]
    hold = ((np.linalg.norm(tip_goal - press_target, axis=1) < 1e-9)
            & (k_gauss >= k_gauss.max() - 1e-9))

    return dict(name=name, t=d['t'] - d['t'][0], force_n=d['force'] * G_TO_N,
                q_motor=d['q_motor'], comp_pos=d['comp_pos'], goal=d['goal'],
                k_gauss=k_gauss, strength=float(k_gauss.max()), hold=hold,
                press_target=press_target)


def backbone(run):
    """FK backbone at the mean steady-state pose."""
    q_ss = run['q_motor'][run['hold']].mean(axis=0)
    return np.array([FK_Multi(q_ss, s)[:3] for s in S_BACKBONE])


## Button pressing


In [ ]:
cfg_b = load_cfg('button')
runs_b = [load_run('button_pressing', n) for n in cfg_b.GAUSS_CONDITIONS]

for r in runs_b:
    f_ss = r['force_n'][r['hold']]
    print(f"{r['name']:>9}  strength {r['strength']:6.1f}  {r['hold'].sum():5d} hold samples"
          f"  force {f_ss.mean():6.2f} +- {f_ss.std():.2f} N")


In [ ]:
fig, ax = plt.subplots(figsize=(FIG_W_DOUBLE, FIG_W_DOUBLE * 0.45))
for i, r in enumerate(runs_b):
    ax.plot(r['t'], r['force_n'], color=COLORS[i], linewidth=1.6,
            label=f"{r['name']} ({r['strength']:.0f})")
    ax.fill_between(r['t'], 0, r['force_n'], where=r['hold'], color=COLORS[i], alpha=0.12, linewidth=0)

ax.set_xlabel('Time [s]')
ax.set_ylabel('Shear force [N]')
ax.set_ylim(min(0.0, min(r['force_n'].min() for r in runs_b)), None)
ax.grid(GRID)
ax.legend(title='Gaussian strength [N/m]', fontsize=11, title_fontsize=11)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'button_pressing', 'force_vs_time.pdf'))
plt.show()


In [ ]:
# Steady-state contact force against Gaussian strength.
strength = np.array([r['strength'] for r in runs_b])
f_mean   = np.array([r['force_n'][r['hold']].mean() for r in runs_b])
f_std    = np.array([r['force_n'][r['hold']].std()  for r in runs_b])
order    = np.argsort(strength)

fig, ax = plt.subplots(figsize=(FIG_W_SINGLE, FIG_W_SINGLE * 0.85))
ax.errorbar(strength[order], f_mean[order], yerr=f_std[order], marker='o',
            color=COLORS[1], linewidth=1.8, markersize=5, capsize=3)
ax.set_xlabel(r'Gaussian strength [N/m]')
ax.set_ylabel('Shear force [N]')
ax.set_xlim(0, None)
ax.set_ylim(min(0.0, f_mean.min()), None)
ax.grid(GRID)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'button_pressing', 'force_vs_strength.pdf'))
plt.show()


In [ ]:
# Steady-state shape, with the two Gaussian attachment points marked.
fig, ax = plt.subplots(figsize=(FIG_W_SINGLE * 1.25, FIG_W_SINGLE * 1.9))
for i, r in enumerate(runs_b):
    b = backbone(r)
    ax.plot(b[:, 0], b[:, 2], color=COLORS[i], linewidth=2.2,
            label=f"{r['name']} ({r['strength']:.0f})", zorder=3)
    for s in (1/3, 2/3):
        p = b[np.argmin(np.abs(S_BACKBONE - s))]
        ax.plot(p[0], p[2], 'o', color=COLORS[i], markersize=5, zorder=4)

ax.plot(*cfg_b.BUTTON_POS[[0, 2]], '*', color='k', markersize=16, label='button', zorder=5)
ax.plot(*cfg_b.OBSTACLE_POS[[0, 2]], 'X', color='k', markersize=11, label='obstacle', zorder=5)
circle = plt.Circle(cfg_b.OBSTACLE_POS[[0, 2]], cfg_b.SIGMA, fill=False,
                    linestyle='--', color='k', linewidth=1.0, alpha=0.5, zorder=2)
ax.add_patch(circle)

ax.set_xlabel('X [m]')
ax.set_ylabel('Z [m]')
ax.set_aspect('equal')
ax.grid(GRID)
ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0), fontsize=11, borderaxespad=0.)
plt.tight_layout()
plt.savefig(os.path.join('outputs', 'button_pressing', 'backbone.pdf'))
plt.show()


## Person reaching

No load cell, so the `force` channel is flat -- the useful quantity is how far the tip is
held back from the target.


In [ ]:
cfg_p = load_cfg('person')
u_p   = (cfg_p.PERSON_TARGET - cfg_p.DEFAULT_START[0])
u_p  /= np.linalg.norm(u_p)
runs_p = [load_run('person_avoiding', n) for n in cfg_p.GAUSS_CONDITIONS]

for r in runs_p:
    tip_ss = r['comp_pos'][r['hold'], 0, :].mean(axis=0)
    # signed along the approach direction: positive = short of the target,
    # negative = pushed past it. An unsigned norm cannot tell those apart.
    r['standoff'] = float((cfg_p.PERSON_TARGET - tip_ss) @ u_p)
    print(f"{r['name']:>9}  strength {r['strength']:6.1f}  standoff {r['standoff']*100:5.1f} cm")


In [ ]:
fig, (ax, ax2) = plt.subplots(1, 2, figsize=(FIG_W_DOUBLE, FIG_W_DOUBLE * 0.55),
                              gridspec_kw=dict(width_ratios=[1.35, 1]))
for i, r in enumerate(runs_p):
    b = backbone(r)
    ax.plot(b[:, 0], b[:, 2], color=COLORS[i], linewidth=2.2,
            label=f"{r['name']} ({r['strength']:.0f})", zorder=3)
    for s in (1/3, 2/3):
        p = b[np.argmin(np.abs(S_BACKBONE - s))]
        ax.plot(p[0], p[2], 'o', color=COLORS[i], markersize=5, zorder=4)

ax.plot(*cfg_p.PERSON_TARGET[[0, 2]], '*', color='k', markersize=16, label='target', zorder=5)
ax.plot(*cfg_p.OBSTACLE_POS[[0, 2]], 'X', color='k', markersize=11, label='obstacle', zorder=5)
ax.set_xlabel('X [m]')
ax.set_ylabel('Z [m]')
ax.set_aspect('equal')
ax.grid(GRID)
ax.legend(fontsize=11, loc='lower right')

s_p = np.array([r['strength'] for r in runs_p])
d_p = np.array([r['standoff'] * 100 for r in runs_p])
o   = np.argsort(s_p)
ax2.plot(s_p[o], d_p[o], marker='o', color=COLORS[1], linewidth=1.8, markersize=5)
ax2.set_xlabel(r'Gaussian strength [N/m]')
ax2.set_ylabel('Tip stand-off [cm]')
ax2.axhline(0, color='k', linewidth=0.8, alpha=0.4)
ax2.set_xlim(0, None)
ax2.set_ylim(min(0.0, d_p.min()), None)
ax2.grid(GRID)

plt.tight_layout()
plt.savefig(os.path.join('outputs', 'person_avoiding', 'standoff.pdf'))
plt.show()
